In [ ]:
import numpy as np
import pickle
from gensim.models import Word2Vec

# --- Load all models ---
with open('../Models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)

with open('../Models/lr_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)

with open('../Models/nb_model.pkl', 'rb') as f:
    nb_model = pickle.load(f)

with open('../Models/rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

w2v_model = Word2Vec.load('../Models/w2v_model.model')

# --- Helper ---
def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

def label(pred):
    return "🚨 Phishing" if pred == 1 else "✅ Safe"

# --- Main prediction function ---
def predict_email(email_text):
    print("\n" + "="*50)
    print("📧 Analyzing Email...")
    print("="*50)

    # TF-IDF transform
    tfidf_vec = tfidf_vectorizer.transform([email_text])

    # Word2Vec transform
    tokens = email_text.split()
    w2v_vec = np.array([get_avg_vector(tokens, w2v_model)])

    # --- Logistic Regression ---
    lr_pred = lr_model.predict(tfidf_vec)[0]
    lr_conf = lr_model.predict_proba(tfidf_vec)[0][lr_pred] * 100
    print(f"\n[Logistic Regression]  {label(lr_pred)}  ({lr_conf:.1f}% confident)")

    # --- Naive Bayes ---
    nb_pred = nb_model.predict(tfidf_vec)[0]
    nb_conf = nb_model.predict_proba(tfidf_vec)[0][nb_pred] * 100
    print(f"[Naive Bayes]          {label(nb_pred)}  ({nb_conf:.1f}% confident)")

    # --- Random Forest ---
    rf_pred = rf_model.predict(w2v_vec)[0]
    rf_conf = rf_model.predict_proba(w2v_vec)[0][rf_pred] * 100
    print(f"[Random Forest]        {label(rf_pred)}  ({rf_conf:.1f}% confident)")

    # --- Majority Vote ---
    votes = [lr_pred, nb_pred, rf_pred]
    final = 1 if sum(votes) >= 2 else 0
    print(f"\n>>> Majority Verdict:  {label(final)}")
    print("="*50)


# --- Try it out ---
email = """
Dear user, your account has been suspended. 
Click here immediately to verify your identity and avoid permanent ban: http://secure-login-verify.com
"""

predict_email(email)